## Jak włączyć obsługę Django w PyCharm?

![](https://intellij-support.jetbrains.com/hc/user_images/eVSRccUymV1BCrO57Gjd-w.png)

## Zadanie

Sprawdź, czy użytkownik jest uwierzytelniony.

Można wykorzystać informacje o użytkowniku z

`{{ user.is_authenticated }}`.

# Django - Model danych
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Po co nam baza danych?
* "Klasyczne" zarządzanie bazami danych
* Tworzenie tabeli w Django
  * Podgląd migracji
  * Migracja
* Aplikowanie migracji
* Przeglądanie bazy danych przy pomocy DB Browser
* Podsumowanie migracji
* Panel administracyjny
* Dalsze kroki
* Dodatkowe materiały

## Po co nam baza danych?

Najprościej mówiąc - to kontener, w którym zapisane będą informacje o użytkownikach naszej strony, filmy, aktualności i wszystkie inne rzeczy, które dodamy. Baza danych w naszym przypadku pozwala na przechowywanie danych, które będą dodawane przez użytkownika aplikacji, a nie przez programistę. My, jako programiści, jedynie ustalimy reguły, wyznaczając modele. Na tych zajęciach skupimy się na bazach relacyjnych, takich jak SQLite3. Warto jednak wiedzieć, że typów baz danych jest wiele, a jednym z takich, który znamy z codzienności, jest arkusz kalkulacyjny.

W tym konspekcie zaczniemy od modelu przeznaczonego do przechowywania filmów.

Zobaczmy jak to wygląda w praktyce...

## "Klasyczne" zarządzanie bazami danych

Gdybyśmy nie mieli narzędzi takich jak Django i innych, praca z bazami danych (w większości) wymagałaby nauczenia się dodatkowego języka: SQL.

Przykładowe tworzenie tabeli z wykorzystaniem SQL:

```sql
CREATE TABLE "movies_movie" (
    "id" integer NOT NULL PRIMARY KEY AUTOINCREMENT,
    "title" varchar(100) NOT NULL,
    "short_description" text NOT NULL,
    "published_at" datetime NOT NULL);
COMMIT;
```

Byłby to kod, który trzeba przetestować i gdzieś zapisać. Następnie należałoby pamiętać, żeby wykonać go na odpowiedniej bazie danych. Gdyby w trakcie projektu wygląd tej tabeli miał się zmienić (a często tak się dzieje!), wymagane byłyby kolejne skrypty napisane w SQL. A to tylko początek problemów...

## Tworzenie tabeli w Django

Tworzenie tabeli z wykorzystaniem Django (w pliku [`models.py`](http://localhost:8888/edit/movies/models.py)):

```python
class Movie(models.Model):
    title = models.CharField(max_length=100)
    short_description = models.TextField()
    published_at = models.DateTimeField()
```

### Podgląd migracji

Jeśli chcielibyśmy jedynie podejrzeć co zostanie wygenerowane możemy podczas migracji wykorzystać flagi:

```sh
--dry-run -v 3
```

Pierwsza część pozwala na "wykonanie na sucho", czyli cała maszyneria zostanie uruchomiona, ale migracja nie zostanie zapisana na dysku/stworzona. `-v 3` natomiast pozwala na wypisanie wszystkich szczegółów procesu generowania migracji a także samej migracji.

 Wynik działania tego polecenia możemy zobaczyć na kolejny slajdzie.

In [20]:
# Poniższe polecenie wypisze na konsole poniższe informacje

!python3 manage.py makemigrations --dry-run -v 3

Migrations for 'movies':
  movies/migrations/0001_initial.py
    + Create model Movie
Full migrations file '0001_initial.py':
# Generated by Django 5.2 on 2025-05-17 14:52

from django.db import migrations, models


class Migration(migrations.Migration):

    initial = True

    dependencies = [
    ]

    operations = [
        migrations.CreateModel(
            name='Movie',
            fields=[
                ('id', models.BigAutoField(auto_created=True, primary_key=True, serialize=False, verbose_name='ID')),
                ('title', models.CharField(max_length=100)),
                ('short_description', models.TextField()),
                ('published_at', models.DateTimeField()),
            ],
        ),
    ]


Jak widzimy, najpierw narzędzie poinformowało nas, że tworzona zostaje migracja dla aplikacji "movies" a następnie w terminalu wypisana została cała migracja.

To jest ten krótki kawałek kodu w Pythonie począwszy od
```python
# Generated by Django
```
do samego końca.

### Migracja

Poniższym poleceniem możemy wygenerować migrację. Migracja w tym przypadku to skrypt w Pythonie, który pozwala na utworzenie potrzebnych tabel w bazie danych, która jest podpięta do aplikacji. W naszym przypadku tabele zostaną stworzone w bazie `db.sqlite3`, która znajduje się domyślnie w głównym katalogu całego projektu/repozytorium kodu.

In [24]:
!python3 manage.py makemigrations

It is impossible to add a non-nullable field 'rating' to movie without specifying a default. This is because the database needs something to populate existing rows.
Please select a fix:
 1) Provide a one-off default now (will be set on all existing rows with a null value for this column)
 2) Quit and manually define a default value in models.py.
Select an option: ^C

Cancelled.


Po wykonaniu powyższego polecenia w katalogu `migrations` pojawi się nowy plik nazwany `0001_initial.py`.

In [22]:
!ls -alh movies/migrations

total 8
drwxr-xr-x   5 miklesz  staff   160B May 17 16:57 .
drwxr-xr-x  11 miklesz  staff   352B May 17 16:57 ..
-rw-r--r--   1 miklesz  staff   777B May 17 16:57 0001_initial.py
-rw-r--r--   1 miklesz  staff     0B Apr 27 16:39 __init__.py
drwxr-xr-x   7 miklesz  staff   224B May 17 15:26 __pycache__


Migracje są automatycznie numerowane, natomiast ich nazwę można ustawić:

Polecenie:
```sh
python3 manage.py makemigrations -n nazwa
```
wygeneruje `numer_nazwa.py`.

Na pierwszy rzut oka może się wydawać, że kodu/pracy wcale nie jest mniej. Jednak kluczowe zyski z korzystania z frameworku to:

* nie trzeba się uczyć dodatkowego języka (SQL),

* framework jest dobrze przetestowany przez twórców i użytkowników, więc możemy założyć, że działa,

* przy każdej zmianie w modelu wystarczy uruchomić polecenie w konsoli, a SQL zostanie wygenerowany automatycznie - nie musimy się zastanawiać co się zmieniło.

Innymi słowy - oszczędzamy czas dzięki gotowym narzędziom. 🙂

## Aplikowanie migracji

Teraz kiedy mamy już napisany podstawowy model oraz wygenerowaną migrację trzeba ją zaaplikować na bazę danych.

Utworzy to odpowiednią tabele na nasze filmy w bazie danych.

In [23]:
!python3 manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, contenttypes, movies, sessions
Running migrations:
  Applying movies.0001_initial... OK


W outpucie, w linijce `Applying movies... OK` widnieje nasza migracja! Ona właśnie została zaaplikowana.

## Przeglądanie bazy danych przy pomocy DB Browser

Do przeglądania bazy danych SQLite można wykorzystać [https://sqlitebrowser.org/](https://sqlitebrowser.org/).

Jest to mały darmowy program, który teraz możemy uruchomić i otworzyć nim plik `db.sqlite3`, a w nim, podglądać (przykładowo) niedawno stworzoną tabelę `movies_movie`.

## Podsumowanie migracji

Reasumując, aby przechowywać informacje w bazie danych musimy:

1. Opisać w Django (plik `models.py`) jak ma się nazywać model, jakie ma mieć pola oraz jakie właściwości mają te pola posiadać. (np. pole "data_utworzenia", typ: data, wymagane)
2. Wygenerować migrację (polecenie: `python3 manage.py makemigrations`)
3. Zaaplikować migrację na bazie danych, co spowoduje utworzenie odpowiednich tabel lub wprowadzenie zmian w istniejących (polecenie: `python3 manage.py migrate`)

## Panel administracyjny

Najszybszym (i dla niektórych najwygodniejszym) sposobem, żeby dodać nowe filmy będzie wykorzystanie panelu administracyjnego Django. Jest to narzędzie dostarczone z frameworkiem, domyślnie włączone.

Aby z niego skorzystać musimy zarejestrować nasz model `Movie` jako obsługiwany przez panel administracyjny. W tym celu otwórzmy plik [`admin.py`](http://localhost:8888/edit/movies/admin.py) znajdujący się w katalog aplikacji ([`movies/admin.py`](http://localhost:8888/edit/movies/admin.py)). Znajdziemy tutaj jedynie jeden import i komentarz.

Dopiszmy więc wymagany kod:

```python
from django.contrib import admin
from movies.models import Movie  # NOWE


# Register your models here.
admin.site.register(Movie)  # NOWE
```

W ten sposób aplikacja panelu administracyjnego będzie wiedziała, że istnieje model `Movie` i ma być on zarządzany przez panel.

[Uruchommy więc aplikację](0_Run.ipynb) i przejdźmy pod [http://127.0.0.1:8000/admin/](http://127.0.0.1:8000/admin/)

_Przypomnienie_:

```sh
# polecenie do uruchomienia
python3 manage.py runserver

# polecenie do utworzenia super-użytkownika/admina
python3 manage.py createsuperuser
```

## Dalsze kroki

1. Dodać kilka filmów
2. Otworzyć bazę danych za pomocą programu DB Browser i przeglądnąć dane w tabeli `movies_movie`
3. Konfiguracja etykiety dla obiektów na liście przez użycie metody `__str__` w klasie modelu, np.:
```python
    def __str__(self):
        return self.title
```
4. Dodanie kolejnych pól do modelu, z nowymi typami danych, np.:
```python
    rating = models.IntegerField()
```

## Dodatkowe materiały

* Spis dostępnych pól, które można dodać do modelu: [https://docs.djangoproject.com/en/5.1/ref/models/fields/#field-types](https://docs.djangoproject.com/en/5.1/ref/models/fields/#field-types) 
* Django Cheat Sheet, czyli spis często używanych poleceń i kawałków kodu: [https://github.com/lucrae/django-cheat-sheet](https://github.com/lucrae/django-cheat-sheet)
* Wytłumaczenie czym jest model dla osób mniej zaznajomionych z programowaniem: [https://tutorial.djangogirls.org/pl/django_models/](https://tutorial.djangogirls.org/pl/django_models/)

# Django - Prezentacja danych
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Szablony HTML w Django - szablon bazowy dla naszej aplikacji
* Prezentacja danych z bazy
  * Widok listy filmów
  * Rejestracja podstrony listy filmów
  * Szablon HTML listy
* Dalsze prace

## Szablony HTML w Django - szablon bazowy dla naszej aplikacji

Każda strona WWW ma kilka stałych elementów interfejsu, takich jak: menu, stopka, obszar gdzie wyświetlane są treści, czasem pasek boczny i inne. Każda podstrona naszej aplikacji musi więc składać się z tego samego kodu, kopiowanego wszędzie i trudnego w utrzymaniu. Gdy zrobimy gdzieś literówkę, to musimy poprawiać we wszystkich plikach z osobna! Gdy chcemy dodać coś nowego do interfejsu - też.

Na szczęście istnieją frameworki! Django jak wiele innych narzędzi ma sposób na taką redundancję kodu - [Django HTML Templating Engine](https://docs.djangoproject.com/en/5.2/topics/templates/). W skrócie nazywane dalej szablonami HTML Django.

Do tej pory mieliśmy do czynienia głównie ze standardową składnią HTML.

Django wzbogaca możliwości HTML między innymi o dziedziczenie szablonów. To pozwala nam napisać jeden raz szablon bazowy naszej aplikacji, który będzie dynamicznie załączany do każdej podstrony, która po nim dziedziczy. Brzmi magicznie? Nie szkodzi, zobaczmy jak to wygląda w praktyce.

### Szablon bazowy dla naszej aplikacji

Na dobry początek utwórzmy w katalogu [`movies/templates/`](movies/templates/) nowy plik [`base.html`](http://localhost:8888/edit/movies/templates/base.html). Będzie to nasz bazowy szablon wykorzystywany przez inne podstrony.

In [7]:
!touch movies/templates/base.html

Zacznijmy od czegoś prostego (proponowana początkowa zawartość [`base.html`](http://localhost:8888/edit/movies/templates/base.html)):

```html
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <title>Moja biblioteka filmów</title>
</head>
<body>
    
</body>
</html>
```

Powyższy HTML wyświetli nam jedynie pustą stronę z tytułem karty "Moja biblioteka filmów". Dodajmy więc przynajmniej jakiś nagłówek (oczywiście w środku elementu `<body> </body>`):

```html
<h1>
    Witamy w filmotece!
</h1>
```

W ten sposób będziemy wiedzieli, że każda strona wykorzystuje ten kod HTML, bo nam się wyświetli nagłówek.

Następnie dodajmy HTMLowy tag Django:

```django
{% block content %}
    Tu będzie treść...
{% endblock %}
```

Ten fragment kodu informuje Django, że w tym miejscu pliku HTML można wstawić jakąś zawartość w szablonach HTML, które rozszerzają ten plik. Kontynuujmy...

_Tag `{% block %}{% endblock %}` może być pusty, ale jeśli coś w nim umieścimy zadziała to jak domyślna wartość._

W szablonie z poprzednich zajęć [`hello.html`](http://localhost:8888/edit/movies/templates/hello.html) dodajmy na górze pliku, w pierwszej linijce `extends`.

```django
{% extends "base.html" %}

<h1>Witaj świecie z HTML!</h1>
```

A następnie wejdźmy na [http://127.0.0.1:8000/hello/](http://127.0.0.1:8000/hello/) w przeglądarce. Rezultaty? :)

To teraz zobaczymy co oznaczało "wstawić jakąś zawartość w szablonach HTML, które rozszerzają ten ([`base.html`](http://localhost:8888/edit/movies/templates/base.html))". Zmieńmy kod [`hello.html`](http://localhost:8888/edit/movies/templates/hello.html) na następujący.

```django
{% extends "base.html" %}

{% block content %}
    <h1>Witaj świecie z HTML!</h1>
{% endblock %}
```

I ponownie udajmy się do przeglądarki.

## Prezentacja danych z bazy

W tej chwili mamy działający model na filmy oraz narzędzie, które pozwala na zarządzanie nimi (panel administracyjny). Jednak chcielibyśmy pochwalić się światu zawartością naszej biblioteczki. W tym celu oczywiście nie rozdamy wszystkim haseł dostępu, lub go nie zdejmiemy, to byłoby niebezpieczne!

Spróbujmy zatem wyświetlić listę filmów korzystając z szablonów HTML Django. To one pozwalają nam dynamicznie dodawać treść pobraną z bazy danych (a w międzyczasie zmienianą przez użytkowników w panelu administracyjnym).